In [1]:
import sys
from pathlib import Path
sys.path.append("../src")

from mortgage_agent.schemas import Applicant
from mortgage_agent.calculators import ltv, dti, funds_to_close, usable_assets, reserves_months
from mortgage_agent.rules import Finding, decide, check_credit_score, check_dti, check_bankruptcy, check_foreclosure
sarah = Applicant.model_validate_json(Path("../data/applicants/A007.json").read_text())

print(f"LTV: {ltv(sarah):.1f}%")
print(f"DTI: {dti(sarah):.1f}%")
print(f"Funds to close: ${funds_to_close(sarah):,.0f}")
print(f"Usable assets:  ${usable_assets(sarah):,.0f}")
print(f"Reserves:       {reserves_months(sarah):.1f} months")

LTV: 80.0%
DTI: 35.2%
Funds to close: $89,000
Usable assets:  $55,000
Reserves:       -14.0 months


In [2]:
low_score = Finding(rule_id="B3-5.1-01", severity="decline", reason="Credit score 580 is below the 620 minimum.")
no_letter = Finding(rule_id="B3-4.3-04", severity="condition", reason="Gift of $10,000 has no gift letter.")

print(decide([]).decision)
print(decide([no_letter]).decision)
print(decide([low_score, no_letter]))

approve
approve with conditions
decision='decline' reasons=['Credit score 580 is below the 620 minimum.'] rule_ids=['B3-5.1-01']


In [3]:
print(check_credit_score(sarah))

ben = sarah.model_copy(deep=True)
ben.credit.score = 580
print(check_credit_score(ben))

None
rule_id='B3-5.1-01' severity='decline' reason='Credit score 580 is below the 620 minimum.'


In [4]:
print(check_dti(sarah))

heavy = sarah.model_copy(deep=True)
heavy.loan.underwriting = "manual"
heavy.income.verified_monthly_income = 5_000
print(check_dti(heavy))

gray = sarah.model_copy(deep=True)
gray.loan.underwriting = "manual"
gray.income.verified_monthly_income = 7_000
try:
    check_dti(gray)
except ValueError as e:
    print("Refused:", e)

gray.loan.underwriting = "DU"
print(check_dti(gray))

None
rule_id='B3-6-02' severity='decline' reason='DTI 59.8% exceeds the 45% maximum for manual underwriting.'
Refused: Manual DTI 42.7% is in the 36-45% band, which needs the Eligibility Matrix (not in B3).
None


In [5]:
from datetime import date

print(check_bankruptcy(sarah))

b = sarah.model_copy(deep=True)
b.credit.bankruptcy_chapter = 7
b.credit.bankruptcy_discharge_date = date(2022, 10, 30)
print(check_bankruptcy(b))

b.credit.bankruptcy_discharge_date = date(2022, 10, 31)
print(check_bankruptcy(b))

None
None
rule_id='B3-5.3-07' severity='decline' reason='Chapter 7 bankruptcy discharged 2022-10-31; the 4-year wait ends 2026-10-31, after the 2026-10-30 closing.'


In [6]:
f = sarah.model_copy(deep=True)
f.credit.foreclosure_date = date(2019, 10, 30)
print(check_foreclosure(f))

f.credit.foreclosure_date = date(2020, 3, 15)
print(check_foreclosure(f))

None
rule_id='B3-5.3-07' severity='decline' reason='Foreclosure completed 2020-03-15; the 7-year wait ends 2027-03-15, after the 2026-10-30 closing.'
